# 第 8 章：工程部署及性能分析 — 章节实践与测试

## 小节概述

本小节包含两部分内容：

1. **综合编程实践**：独立完成注意力算子的向量化改造（Matmul+Softmax 融合的思路验证）；
2. **知识测验**：覆盖 msOpGen / 打包部署 / msProf 报告字段 / msSanitizer / msDebug。

完成所有题目后，可运行最后的 cell 查看参考答案。


## 一、综合编程实践题

### 实践题：注意力算子的向量化改造

08.02 中我们使用了**纯标量实现**（GM 标量访问 + UB 数组），性能瓶颈是标量 GM 访问延迟
（msProf 中 `aiv_vec_ratio ≈ 0`）。请基于本章工程完成一次**向量化改造**：

**任务 1（必做）**：在 08.02 工程中，将**阶段 B（o = P × V）的内层 k 循环**改造成
`DataCopy` 批量读入 V 行 + `Muls`/`Add` 向量指令累加（一次处理 64 个元素，代替逐元素 GM 标量读）。

提示：参考 08.02 Kernel 的 UB 数组结构，注意改造后 `aiv_vec_ratio` 应显著上升。

```cpp
// 伪代码：o 行 j 方向向量化（dim=64 一次算完一行）
// 1) oRow[0..63] 清零
// 2) for k in 0..S-1:
//      从 v[k][0..63] 批量读入 UB（DataCopy）
//      oRow[j] += P[k] * v[k][j]   （Muls + Add 向量化）
```

**任务 2（选做）**：将**阶段 A** 的 QKᵀ 同样向量化（逐 k 累加 kt 行），并重新运行
`mssanitizer -t memcheck` 确认无越界。

**验证标准**：改造后 benchmark `result: PASS`，且 msProf 中 `aiv_vec_ratio` 明显大于 0。


In [ ]:
# ===== 实践题入口 =====
# 1) 在 src/attention_op/custom_ops/generated/AttentionCustom/op_kernel/attention_custom.cpp
#    中改造阶段 B 的 k 循环（向量化）
# 2) 重新编译安装 + 运行验证：
#    bash scripts/build_ops.sh
#    source scripts/env_custom_opp.sh
#    aclnn_runner/build/main_attention_benchmark data 512 64
# 3) 采集性能并对比 aiv_vec_ratio 变化：
#    bash scripts/run_profiling.sh 512 --output prof
#
# 本 cell 检查当前实现是否已向量化（未改造时 aiv_vec_ratio≈0）：
import glob, csv, subprocess, os

prof = glob.glob('src/attention_op/prof/prof_512/OPPROF_*')
if not prof:
    print('请先完成 08.02 步骤 5 的 msProf 采集')
else:
    with open(f'{prof[-1]}/PipeUtilization.csv') as f:
        r = next(csv.DictReader(f))
    print(f'aiv_vec_ratio = {r.get("aiv_vec_ratio")}  （改造前 ≈ 0）')
    print('改造后该值应显著上升 —— 用于判断是否完成向量化。')

## 二、选择题

### 题目 1：msOpGen 生成算子工程使用的命令是？

A. `msopgen gen -i ops.json -c ai_core-ascend910b -out <dir>`
B. `msopgen build -i ops.json -o <dir>`
C. `msprof op -i ops.json`
D. `msopgen gen -o ops.json -c <dir>`


In [ ]:
# 请将 answer1 修改为你的答案（A/B/C/D）
answer1 = ''
print(f'题目1 你的答案：{answer1}')


### 题目 2：自定义算子包安装到用户目录的命令是？

A. `bash xxx.run --install-path=${HOME}`（安装到 `${HOME}/vendors/customize`）
B. `pip install xxx.run`
C. `msopgen install xxx.run`
D. `cmake --install .`


In [ ]:
answer2 = ''  # 请修改为你的答案（A/B/C/D）
print(f'题目2 你的答案：{answer2}')


### 题目 3：msProf 报告中表示算子总执行时间的字段是？

A. `aiv_vec_ratio`
B. `Task Duration(us)`
C. `Block Dim`
D. `aic_time(us)`


In [ ]:
answer3 = ''  # 请修改为你的答案（A/B/C/D）
print(f'题目3 你的答案：{answer3}')


### 题目 4：08.02 实测中 seq_len 从 512 翻倍到 1024，耗时约从 85ms 变为？

A. ~170ms（线性）
B. ~355ms（×4，O(S²)）
C. ~85ms（不变）
D. ~21ms（÷4）


In [ ]:
answer4 = ''  # 请修改为你的答案（A/B/C/D）
print(f'题目4 你的答案：{answer4}')


### 题目 5：mssanitizer 中用于内存越界检测的工具是？

A. `-t racecheck`
B. `-t initcheck`
C. `-t memcheck`
D. `-t synccheck`


In [ ]:
answer5 = ''  # 请修改为你的答案（A/B/C/D）
print(f'题目5 你的答案：{answer5}')


## 三、填空题

### 题目 6：自定义算子通过 ______ 接口实现单算子调用（08.02 中为 `aclnnAttentionCustom`）。


In [ ]:
answer6 = ''  # 请填写你的答案
print(f'题目6 你的答案：{answer6}')


### 题目 7：注意力算子总计算量约 ______ FLOPs（以 S、D 表示），复杂度 O(S²)。


In [ ]:
answer7 = ''  # 请填写你的答案
print(f'题目7 你的答案：{answer7}')


### 题目 8：msDebug 需要在驱动开启 ______ 通道后才能断点调试（云环境可能不支持）。


In [ ]:
answer8 = ''  # 请填写你的答案
print(f'题目8 你的答案：{answer8}')


## 四、查看参考答案

完成所有题目后，运行以下 cell 查看实践题参考答案（知识测验答案见 `answer/chapter_test_answer.md`）：

In [ ]:
# 查看参考答案（实践题）
import os, subprocess

NOTES = os.path.abspath('.')
print('参考实现位于：')
print('  answer/practice_vectorize_answer.md （向量化改造思路与核心代码）')
subprocess.run(['sed', '-n', '1,40p', 'answer/practice_vectorize_answer.md'])
